# Promptable segmentation with SAM 3

Segment every object matching a concept (a short text phrase or a bounding box) using Meta's [Segment Anything Model 3](https://huggingface.co/facebook/sam3).

**What's in this recipe:**

- Segment a single object from a text prompt
- Segment many instances of a concept at once
- Segment a specific region with a bounding box
- Segment objects across the frames of a video
- Erase segmented objects with Hugging Face inpainting

## Problem

Classical segmentation models output a fixed set of class labels. SAM 3 instead performs *promptable concept segmentation*: you describe what you want with a short noun phrase ("zebra", "car") or a bounding box, and the model returns one instance mask for every matching object.

| Approach | Output | Prompting |
|----------|--------|-----------|
| Panoptic segmentation (DETR) | Fixed COCO classes | None |
| Promptable segmentation (SAM 3) | Per-instance binary masks | Text and/or boxes |

## Solution

Use the `sam_for_segmentation` UDF in `pixeltable.functions.huggingface`. It returns a typed result with one entry per detected instance: a confidence `score`, a bounding `box`, and a binary `mask`. The `overlay_segmentation` vision UDF accepts those masks directly, so you can visualize results without any glue code.

> **Note:** `facebook/sam3` is a gated model. Request access on its [model page](https://huggingface.co/facebook/sam3), then authenticate with `huggingface-cli login` (or set the `HF_TOKEN` environment variable) before running this notebook.

In [ ]:
%pip install -qU pixeltable torch transformers

### Set up a table of images

We load two images: a single grazing zebra and a bento box full of food. A Pixeltable table stores the originals; every example below computes its segmentation on the fly with `select`, so nothing extra is persisted.

In [ ]:
import pixeltable as pxt
from pixeltable.functions.huggingface import sam_for_segmentation
from pixeltable.functions.vision import overlay_segmentation

pxt.drop_dir('sam3_demo', force=True)
pxt.create_dir('sam3_demo')

base_url = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/images'
zebra = f'{base_url}/000000000034.jpg'
bento = f'{base_url}/000000000009.jpg'

images = pxt.create_table('sam3_demo/images', {'image': pxt.Image})
images.insert([{'image': zebra}, {'image': bento}])

### Segment from a text prompt

Pass a short noun phrase as `text`. SAM 3 returns one binary mask per matching instance. Here we segment the zebra and overlay the mask on the original image with `overlay_segmentation`.

In [ ]:
images.select(
    images.image,
    segmented=overlay_segmentation(
        images.image,
        sam_for_segmentation(images.image, text='zebra').masks,
        draw_contours=True,
    ),
).where(images.image == zebra).collect()

### Segment many instances at once

A single call finds every instance of a concept. Segmenting `broccoli` in the bento box returns several masks, each drawn in its own color. The `scores` column lists the confidence of each detected instance.

In [ ]:
images.select(
    segmented=overlay_segmentation(
        images.image,
        sam_for_segmentation(images.image, text='broccoli').masks,
        draw_contours=True,
    ),
    scores=sam_for_segmentation(images.image, text='broccoli').scores,
).where(images.image == bento).collect()

### Segment a region with a bounding box

Skip the text prompt and pass a bounding box in `[x1, y1, x2, y2]` pixel coordinates to segment whatever concept that region contains.

In [ ]:
images.select(
    images.image,
    segmented=overlay_segmentation(
        images.image,
        sam_for_segmentation(
            images.image, input_boxes=[[10, 30, 470, 300]]
        ).masks,
        draw_contours=True,
    ),
).where(images.image == zebra).collect()

## Segment objects across a video

Segmentation works on video frames too. We split the first three seconds of a Bangkok traffic clip into frames at the video's native frame rate with `frame_iterator`, segment every `car`, overlay the masks, and reassemble the result with the `make_video` aggregator. Extracting and rebuilding at the same frame rate keeps the output at the original tempo.

SAM segments each frame independently and does not track objects across frames, so the per-instance colors would otherwise flicker from frame to frame. To keep the overlay stable, we merge every car mask in a frame into a single region, so all cars share one consistent color.

In [ ]:
from pixeltable.functions.video import frame_iterator, make_video

pxt.drop_dir('sam3_video', force=True)
pxt.create_dir('sam3_video')

bangkok = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/bangkok.mp4'

videos = pxt.create_table('sam3_video/videos', {'video': pxt.Video})
videos.insert([{'video': bangkok}])

frames = pxt.create_view(
    'sam3_video/frames',
    videos,
    iterator=frame_iterator(videos.video),
)

In [ ]:
@pxt.udf
def merge_masks(
    masks: pxt.Array[(None, None, None), pxt.Bool],
) -> pxt.Array[(None, None, None), pxt.Bool]:
    # Collapse all instance masks into one, so every car gets the same color.
    if masks.shape[0] == 0:
        return masks
    return masks.any(axis=0)[None]

In [ ]:
# bangkok.mp4 plays at 25 fps; rebuild at the same rate to keep the tempo.
frames.where(frames.frame_attrs.time < 3.0).group_by(videos).select(
    segmented_video=make_video(
        frames.pos,
        overlay_segmentation(
            frames.frame,
            merge_masks(
                sam_for_segmentation(frames.frame, text='car').masks
            ),
            draw_contours=True,
        ),
        fps=25,
    ),
).collect()

## Remove objects with inpainting

We can go one step further: segment the cars in a single frame, turn the masks into an inpainting mask, and erase the cars with a Hugging Face inpainting model. SAM 3 decides *what* to remove; a `diffusers` inpainting pipeline decides *how* to fill the hole.

This downloads the open `diffusers/stable-diffusion-xl-1.0-inpainting-0.1` model the first time it runs.

In [ ]:
%pip install -qU diffusers accelerate

In [ ]:
import PIL.Image
import PIL.ImageFilter

from pixeltable.functions.util import resolve_torch_device


@pxt.udf
def car_mask(
    masks: pxt.Array[(None, None, None), pxt.Bool],
) -> PIL.Image.Image:
    # White where any car was detected (the region to erase), black elsewhere.
    union = (masks.any(axis=0) * 255).astype('uint8')
    mask = PIL.Image.fromarray(union, mode='L')
    # Dilate the mask so the whole car footprint (edges, shadows) is covered;
    # a pixel-tight mask leaves car fragments the model rebuilds into wonky cars.
    return mask.filter(PIL.ImageFilter.MaxFilter(25))


_inpaint_pipeline = None


@pxt.udf
def erase(
    image: PIL.Image.Image, mask: PIL.Image.Image, prompt: str
) -> PIL.Image.Image:
    import torch  # noqa: F401
    from diffusers import AutoPipelineForInpainting

    global _inpaint_pipeline
    if _inpaint_pipeline is None:
        device = resolve_torch_device('auto')
        _inpaint_pipeline = AutoPipelineForInpainting.from_pretrained(
            'diffusers/stable-diffusion-xl-1.0-inpainting-0.1'
        ).to(device)

    width, height = 1024, 576
    result = _inpaint_pipeline(
        prompt=prompt,
        negative_prompt='car, vehicle, truck, bus, van',
        image=image.resize((width, height)),
        mask_image=mask.resize((width, height)),
        num_inference_steps=30,
        guidance_scale=8.0,
        strength=0.99,
    ).images[0]
    return result.resize(image.size)

In [ ]:
one_frame = frames.where(frames.pos == 40)

one_frame.select(
    frames.frame,
    cars_removed=erase(
        frames.frame,
        car_mask(
            sam_for_segmentation(frames.frame, text='car').masks
        ),
        prompt='empty asphalt road, smooth grey pavement',
    ),
).collect()

## Explanation

SAM 3 differs from earlier segmentation models in two ways:

1. **Concept-based prompting.** Instead of choosing from a fixed taxonomy, you describe the concept with free-form text, bounding boxes, or both.
2. **Instance masks for every match.** One forward pass returns a mask, score, and box for each matching instance.

Because `sam_for_segmentation` is an ordinary UDF, you can store its result in a computed column to segment new rows automatically, or compute it on the fly with `select` as we did here. `overlay_segmentation` consumes the per-instance mask stack directly, so visualizing results never requires reshaping arrays by hand.

## See also

- [Compare object detection and panoptic segmentation](./img-detection-vs-segmentation)
- [Detect objects in images](./img-detect-objects)
- [SAM 3 documentation](https://huggingface.co/docs/transformers/model_doc/sam3)